# RHINO Spectrometer Comparison — Corrected Science Band
## 60–85 MHz (EoR Global Signal)
### Mbatshi Jerry Junior Mbulawa | Jodrell Bank Observatory | University of Manchester
### Supervised by Dr. Phil Bull

---

**Purpose:** Replot all RHINO spectrometer comparison figures using the correct science band  
(60–85 MHz, EoR global signal) instead of the previously used 1400–1440 MHz band.

**Key reference:** Bull et al. (2025), *RAS Techniques and Instruments*, rzaf046.  
RHINO operates at **60–85 MHz**, targeting the 21cm global signal from Cosmic Dawn  
and the Epoch of Reionisation. The EDGES absorption feature sits at **78 MHz** (z ≈ 17.2).

**Data:** All `.npy` arrays span the full 0–2212 MHz Nyquist band.  
The 60–85 MHz science band is extracted by applying a frequency mask to each array.

---

| Figure | What it shows | Data source |
|--------|--------------|-------------|
| Fig 1  | Theoretical FFT vs PFB filter response | Band-independent |
| Fig 2  | Synthetic EoR signal detection test | Simulation |
| Fig 3  | Wideband spectrum 0–500 MHz, RHINO band highlighted | DS1, DS2, DS3, DS4 |
| Fig 4  | RHINO band zoom 60–85 MHz, FFT vs PFB | DS2, DS3, DS4 |
| Fig 5  | ⚠ Waterfall — REQUIRES NEW OBSERVATIONS | Not available |
| Fig 6  | Hi-res spectrum 60–85 MHz at 4.22 kHz/bin | DS2, DS3 |
| Fig 7  | ⚠ Hi-res waterfall — REQUIRES NEW OBSERVATIONS | Not available |
| Fig 8  | Radiometer equation: noise vs integration depth | DS1, DS2, DS3, DS4, DS4-Long |

**Run all cells top to bottom. Every cell prints PASS, WARN, or SKIP.**

In [1]:
# ═══════════════════════════════════════════════════════════════════
# CELL 1 — Imports
# ═══════════════════════════════════════════════════════════════════
import numpy as np
import matplotlib
matplotlib.use('Agg')          # headless — safe on Mac and RFSoC
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib.gridspec import GridSpec
import os, sys, glob, json, datetime
from scipy import signal as scipy_signal

# Check versions
import numpy, matplotlib, scipy
print('NumPy    :', numpy.__version__)
print('Matplotlib:', matplotlib.__version__)
print('SciPy    :', scipy.__version__)
print('PASS imports OK')

NumPy    : 2.3.4
Matplotlib: 3.10.6
SciPy    : 1.16.3
PASS imports OK


In [2]:
# ═══════════════════════════════════════════════════════════════════
# CELL 2 — Paths and constants
# ═══════════════════════════════════════════════════════════════════

# ── Data directories ─────────────────────────────────────────────
BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

PATHS = {
    'DS1': BASE + '/Jodrell_Discone/No_LNA',   # Discone, no LNA
    'DS2': BASE + '/Jodrell_Discone/LNA',       # Discone + ZKL-2+ LNA
    'DS3': BASE + '/Jodrell_Yagi/LNA',          # Yagi + CobraX LNA
    'DS4': BASE + '/Jodrell_load/LNA',          # Load + ZKL-2+ LNA (~21 min)
    'DS4L': BASE + '/LNA_1hour',                # Load + ZKL-2+ LNA (3 h, checkpoints)
}

# ── Output directory ─────────────────────────────────────────────
OUT = BASE + '/rhino_figures_60_85MHz'
os.makedirs(OUT, exist_ok=True)
print('Output directory:', OUT)

# ── Hardware constants ────────────────────────────────────────────
FS_MHZ          = 4423.680          # ADC sample rate (Msps)
N_FFT_COARSE    = 16384             # Coarse FFT length → 8193 bins
N_FFT_HIRES     = 1048576           # Hi-res FFT length → 524289 bins
N_TAPS          = 4                 # PFB taps
DF_COARSE_KHZ   = FS_MHZ * 1e3 / N_FFT_COARSE   # 270.0 kHz/bin
DF_HIRES_KHZ    = FS_MHZ * 1e3 / N_FFT_HIRES    # 4.219 kHz/bin

# ── Science band (CORRECTED) ──────────────────────────────────────
RHINO_LO_MHZ    = 60.0
RHINO_HI_MHZ    = 85.0
EDGES_MHZ       = 78.0             # EDGES absorption feature centre
FM_LO_MHZ       = 87.5            # FM band lower edge
FM_HI_MHZ       = 108.0           # FM band upper edge

# Radiometer reference sub-band (RFI-quiet region inside science band)
REF_LO_MHZ      = 60.0
REF_HI_MHZ      = 75.0

# ── Plot style ────────────────────────────────────────────────────
plt.rcParams.update({
    'figure.dpi'       : 150,
    'figure.facecolor' : 'white',
    'axes.facecolor'   : '#f8f8f8',
    'axes.grid'        : True,
    'grid.alpha'       : 0.4,
    'font.size'        : 10,
    'axes.titlesize'   : 11,
    'axes.labelsize'   : 10,
    'legend.fontsize'  : 9,
    'lines.linewidth'  : 1.2,
})

COLOURS = {
    'fft' : '#1f77b4',   # blue
    'pfb' : '#d62728',   # red
    'ds1' : '#2ca02c',   # green
    'ds2' : '#ff7f0e',   # orange
    'ds3' : '#9467bd',   # purple
    'ds4' : '#8c564b',   # brown
    'ds4l': '#17becf',   # cyan
}

print('Coarse resolution : %.1f kHz/bin  (%d bins in 60–85 MHz)' %
      (DF_COARSE_KHZ, int((RHINO_HI_MHZ - RHINO_LO_MHZ) * 1e3 / DF_COARSE_KHZ)))
print('Hi-res resolution : %.3f kHz/bin  (%d bins in 60–85 MHz)' %
      (DF_HIRES_KHZ, int((RHINO_HI_MHZ - RHINO_LO_MHZ) * 1e3 / DF_HIRES_KHZ)))
print('PASS configuration OK')

Output directory: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz
Coarse resolution : 270.0 kHz/bin  (92 bins in 60–85 MHz)
Hi-res resolution : 4.219 kHz/bin  (5925 bins in 60–85 MHz)
PASS configuration OK


In [3]:
# ═══════════════════════════════════════════════════════════════════
# CELL 3 — Data loader utility
# Loads whichever files exist; skips gracefully if missing.
# ═══════════════════════════════════════════════════════════════════

def load_npy(path, fname, label=''):
    """Load a .npy file. Returns array or None if missing/unreadable."""
    full = os.path.join(path, fname)
    if not os.path.exists(full):
        print('  SKIP %-55s not found' % fname)
        return None
    try:
        arr = np.load(full, allow_pickle=False)
        if arr.size == 0:
            print('  SKIP %-55s empty array (mask bug)' % fname)
            return None
        tag = (' [%s]' % label) if label else ''
        print('  OK   %-55s shape=%-15s dtype=%s%s' %
              (fname, str(arr.shape), arr.dtype, tag))
        return arr
    except Exception as e:
        print('  FAIL %-55s %s' % (fname, str(e)))
        return None

def glob_load_npy(path, pattern):
    """Load all .npy files matching a glob pattern. Returns sorted list of (N, array)."""
    files = sorted(glob.glob(os.path.join(path, pattern)))
    results = []
    for f in files:
        try:
            arr = np.load(f, allow_pickle=False)
            if arr.size > 0:
                results.append((os.path.basename(f), arr))
        except Exception:
            pass
    return results

def rhino_mask(freq_mhz):
    """Boolean mask for 60–85 MHz science band."""
    return (freq_mhz >= RHINO_LO_MHZ) & (freq_mhz <= RHINO_HI_MHZ)

def ref_mask(freq_mhz):
    """Boolean mask for radiometer reference sub-band (60–75 MHz, RFI-quiet)."""
    return (freq_mhz >= REF_LO_MHZ) & (freq_mhz <= REF_HI_MHZ)

def extract_N(filename):
    """Extract integration depth N from snapshot filename."""
    import re
    # Handles integ_snap_*_N123.npy and checkpoint_N00123.npy
    m = re.search(r'[_N]0*(\d+)\.npy$', filename)
    return int(m.group(1)) if m else 0

def fix_freq_axis(freq_arr):
    """Auto-detect and fix THz-unit frequency axes (DS4-Long bug)."""
    if freq_arr.max() < 10.0:          # stored in THz → convert to MHz
        freq_arr = freq_arr * 1e6
        print('  NOTE frequency axis converted from THz to MHz')
    return freq_arr

def save_fig(fig, name):
    path = os.path.join(OUT, name)
    fig.savefig(path, dpi=150, bbox_inches='tight')
    print('  Saved:', path)
    plt.close(fig)

print('PASS loader utilities defined')

PASS loader utilities defined


In [4]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4 — Load all datasets
# ═══════════════════════════════════════════════════════════════════
print('=' * 65)
print('LOADING DS1 — Discone, No LNA')
print('=' * 65)
P1 = PATHS['DS1']
ds1 = {}
# DS1 has no full-spectrum coarse/hires averaged arrays — only integration snaps
ds1['integ_freq'] = load_npy(P1, 'integ_freq_20250617_211547.npy')
ds1['integ_final'] = load_npy(P1, 'integ_final_20250617_211547.npy')
ds1['snaps_raw'] = glob_load_npy(P1, 'integ_snap_20250617_211547_N*.npy')
print('  DS1 integration snapshots found: %d' % len(ds1['snaps_raw']))

print()
print('=' * 65)
print('LOADING DS2 — Discone + ZKL-2+ LNA')
print('=' * 65)
P2 = PATHS['DS2']
ds2 = {}
ds2['freq_coarse']  = load_npy(P2, 'freq_coarse_20250617_232907.npy')
ds2['freq_hires']   = load_npy(P2, 'freq_hires_20250617_232907.npy')
ds2['fft_coarse']   = load_npy(P2, 'fft_coarse_20250617_232907.npy')
ds2['pfb_coarse']   = load_npy(P2, 'pfb_coarse_20250617_232907.npy')
ds2['fft_hires']    = load_npy(P2, 'fft_hires_20250617_232907.npy')
ds2['integ_freq']   = load_npy(P2, 'integ_freq_20250617_225835.npy')
ds2['integ_final']  = load_npy(P2, 'integ_final_20250617_225835.npy')
ds2['integ_spec2']  = load_npy(P2, 'integ_spectrum_20250617_232907.npy')
ds2['snaps_raw']    = glob_load_npy(P2, 'integ_snap_20250617_225835_N*.npy')
print('  DS2 integration snapshots found: %d' % len(ds2['snaps_raw']))

print()
print('=' * 65)
print('LOADING DS3 — Yagi + CobraX LNA')
print('=' * 65)
P3 = PATHS['DS3']
ds3 = {}
ds3['freq_coarse']  = load_npy(P3, 'freq_coarse_20250619_200339.npy')
ds3['freq_hires']   = load_npy(P3, 'freq_hires_20250619_200339.npy')
ds3['fft_coarse']   = load_npy(P3, 'fft_coarse_20250619_200339.npy')
ds3['pfb_coarse']   = load_npy(P3, 'pfb_coarse_20250619_200339.npy')
ds3['fft_hires']    = load_npy(P3, 'fft_hires_20250619_200339.npy')
ds3['integ_freq']   = load_npy(P3, 'integ_freq_20250619_193106.npy')
ds3['integ_final']  = load_npy(P3, 'integ_final_20250619_193106.npy')
ds3['snaps_raw']    = glob_load_npy(P3, 'integ_snap_20250619_193106_N*.npy')
print('  DS3 integration snapshots found: %d' % len(ds3['snaps_raw']))

print()
print('=' * 65)
print('LOADING DS4 — Load + ZKL-2+ LNA (~21 min)')
print('=' * 65)
P4 = PATHS['DS4']
ds4 = {}
ds4['freq_coarse']  = load_npy(P4, 'freq_coarse_20250619_211156.npy')
ds4['freq_hires']   = load_npy(P4, 'freq_hires_20250619_211156.npy')
ds4['fft_coarse']   = load_npy(P4, 'fft_coarse_20250619_211156.npy')
ds4['pfb_coarse']   = load_npy(P4, 'pfb_coarse_20250619_211156.npy')
ds4['fft_hires']    = load_npy(P4, 'fft_hires_20250619_211156.npy')
ds4['integ_freq']   = load_npy(P4, 'integ_freq_20250619_204116.npy')
ds4['integ_final']  = load_npy(P4, 'integ_final_20250619_204116.npy')
ds4['snaps_raw']    = glob_load_npy(P4, 'integ_snap_20250619_204116_N*.npy')
print('  DS4 integration snapshots found: %d' % len(ds4['snaps_raw']))

print()
print('=' * 65)
print('LOADING DS4-Long — Load + ZKL-2+ LNA (3 hour, checkpoints)')
print('=' * 65)
P4L = PATHS['DS4L']
ds4l = {}
ds4l['freq_coarse_raw'] = load_npy(P4L, 'freq_coarse.npy')
ds4l['final_mean']      = load_npy(P4L, 'final_mean_coarse.npy')
ds4l['checkpoints']     = glob_load_npy(P4L, 'checkpoint_N*.npy')
if ds4l['freq_coarse_raw'] is not None:
    ds4l['freq_coarse'] = fix_freq_axis(ds4l['freq_coarse_raw'].copy())
print('  DS4-Long checkpoints found: %d' % len(ds4l['checkpoints']))

print()
print('PASS all datasets loaded')

LOADING DS1 — Discone, No LNA
  OK   integ_freq_20250617_211547.npy                          shape=(524289,)       dtype=float64
  OK   integ_final_20250617_211547.npy                         shape=(524289,)       dtype=float32
  DS1 integration snapshots found: 10

LOADING DS2 — Discone + ZKL-2+ LNA
  OK   freq_coarse_20250617_232907.npy                         shape=(8193,)         dtype=float64
  OK   freq_hires_20250617_232907.npy                          shape=(524289,)       dtype=float64
  OK   fft_coarse_20250617_232907.npy                          shape=(8193,)         dtype=float32
  OK   pfb_coarse_20250617_232907.npy                          shape=(8193,)         dtype=float32
  OK   fft_hires_20250617_232907.npy                           shape=(524289,)       dtype=float32
  OK   integ_freq_20250617_225835.npy                          shape=(524289,)       dtype=float64
  OK   integ_final_20250617_225835.npy                         shape=(524289,)       dtype=float32
  OK 

In [5]:
# ═══════════════════════════════════════════════════════════════════
# CELL 4b — Load new datasets (DS5: Discone+LNA 22 Jun, DS6: Discone No-LNA 22 Jun)
# ═══════════════════════════════════════════════════════════════════
print('=' * 65)
print('LOADING DS5 — Discone + ZKL-2+ LNA  (22 Jun 2025, new run)')
print('=' * 65)
P5 = '/Users/user/Downloads/Manny-Masters/Project/Data/Jodrell_Discone_New/LNA'
ds5 = {}
ds5['freq_coarse']  = load_npy(P5, 'freq_coarse_20250622_030852.npy')
ds5['freq_hires']   = load_npy(P5, 'freq_hires_20250622_030852.npy')
ds5['fft_coarse']   = load_npy(P5, 'fft_coarse_20250622_030852.npy')
ds5['pfb_coarse']   = load_npy(P5, 'pfb_coarse_20250622_030852.npy')
ds5['fft_hires']    = load_npy(P5, 'fft_hires_20250622_030852.npy')
ds5['integ_freq']   = load_npy(P5, 'integ_freq_20250622_023735.npy')
ds5['integ_final']  = load_npy(P5, 'integ_final_20250622_023735.npy')
ds5['snaps_raw']    = glob_load_npy(P5, 'integ_snap_20250622_023735_N*.npy')
print('  DS5 integration snapshots found: %d' % len(ds5['snaps_raw']))

print()
print('=' * 65)
print('LOADING DS6 — Discone, No LNA  (22 Jun 2025, new run)')
print('=' * 65)
P6 = '/Users/user/Downloads/Manny-Masters/Project/Data/Jodrell_Discone_New/No_LNA'
ds6 = {}
ds6['freq_coarse']  = load_npy(P6, 'freq_coarse_20250622_040048.npy')
ds6['freq_hires']   = load_npy(P6, 'freq_hires_20250622_040048.npy')
ds6['fft_coarse']   = load_npy(P6, 'fft_coarse_20250622_040048.npy')
ds6['pfb_coarse']   = load_npy(P6, 'pfb_coarse_20250622_040048.npy')
ds6['fft_hires']    = load_npy(P6, 'fft_hires_20250622_040048.npy')
ds6['integ_freq']   = load_npy(P6, 'integ_freq_20250622_033022.npy')
ds6['integ_final']  = load_npy(P6, 'integ_final_20250622_033022.npy')
ds6['snaps_raw']    = glob_load_npy(P6, 'integ_snap_20250622_033022_N*.npy')
print('  DS6 integration snapshots found: %d' % len(ds6['snaps_raw']))
print()
print('PASS DS5 and DS6 loaded')


LOADING DS5 — Discone + ZKL-2+ LNA  (22 Jun 2025, new run)
  OK   freq_coarse_20250622_030852.npy                         shape=(8193,)         dtype=float64
  OK   freq_hires_20250622_030852.npy                          shape=(524289,)       dtype=float64
  OK   fft_coarse_20250622_030852.npy                          shape=(8193,)         dtype=float32
  OK   pfb_coarse_20250622_030852.npy                          shape=(8193,)         dtype=float32
  OK   fft_hires_20250622_030852.npy                           shape=(524289,)       dtype=float32
  OK   integ_freq_20250622_023735.npy                          shape=(524289,)       dtype=float64
  OK   integ_final_20250622_023735.npy                         shape=(524289,)       dtype=float32
  DS5 integration snapshots found: 30

LOADING DS6 — Discone, No LNA  (22 Jun 2025, new run)
  OK   freq_coarse_20250622_040048.npy                         shape=(8193,)         dtype=float64
  OK   freq_hires_20250622_040048.npy                   

In [6]:
# ═══════════════════════════════════════════════════════════════════
# CELL 5 — Build frequency axes and science-band masks
# ═══════════════════════════════════════════════════════════════════

# Canonical frequency axes (recomputed from hardware if loaded arrays differ)
freq_c = np.fft.rfftfreq(N_FFT_COARSE, d=1.0/FS_MHZ)   # (8193,) MHz
freq_h = np.fft.rfftfreq(N_FFT_HIRES,  d=1.0/FS_MHZ)   # (524289,) MHz

# If a dataset has its own saved freq axis, use it (they should be identical)
# but fall back to canonical if missing
def get_freq(ds, key, canonical):
    if ds.get(key) is not None:
        f = fix_freq_axis(ds[key].copy())
        if abs(f.max() - canonical.max()) < 1.0:
            return f
    return canonical

# Science band masks on coarse and hires axes
mask_c  = rhino_mask(freq_c)    # ~92 bins
mask_h  = rhino_mask(freq_h)    # ~5926 bins
ref_c   = ref_mask(freq_c)
ref_h   = ref_mask(freq_h)

f_rhino_c = freq_c[mask_c]      # 60–85 MHz coarse axis
f_rhino_h = freq_h[mask_h]      # 60–85 MHz hires axis

# Redshift axis helper
def freq_to_z(f_mhz):
    """Convert observed frequency (MHz) to 21cm redshift."""
    nu21 = 1420.405751786
    return nu21 / f_mhz - 1.0

z_lo = freq_to_z(RHINO_HI_MHZ)  # z at 85 MHz
z_hi = freq_to_z(RHINO_LO_MHZ)  # z at 60 MHz
z_edges = freq_to_z(EDGES_MHZ)   # z at 78 MHz

print('Science band  : %.0f – %.0f MHz' % (RHINO_LO_MHZ, RHINO_HI_MHZ))
print('Redshift range: z = %.1f – %.1f' % (z_lo, z_hi))
print('EDGES at 78 MHz: z = %.1f' % z_edges)
print('Coarse bins in band : %d  (%.1f kHz/bin)' % (mask_c.sum(), DF_COARSE_KHZ))
print('Hires bins in band  : %d  (%.3f kHz/bin)' % (mask_h.sum(), DF_HIRES_KHZ))
print('PASS frequency axes and masks OK')

Science band  : 60 – 85 MHz
Redshift range: z = 15.7 – 22.7
EDGES at 78 MHz: z = 17.2
Coarse bins in band : 92  (270.0 kHz/bin)
Hires bins in band  : 5926  (4.219 kHz/bin)
PASS frequency axes and masks OK


In [7]:
# ═══════════════════════════════════════════════════════════════════
# CELL 6 — FIG 1: Theoretical FFT vs PFB filter response
# Band-independent — valid as-is, no correction needed.
# ═══════════════════════════════════════════════════════════════════
print('Generating Fig 1: Filter response...')

N_filt  = N_FFT_COARSE
n_taps  = N_TAPS
n_bins  = N_filt // 2 + 1

# ── FFT channel response (Hann window) ───────────────────────────
norm_freq = np.linspace(-4, 4, 8000)   # bins relative to centre
hann_resp = np.sinc(norm_freq) * np.cos(np.pi * norm_freq / N_filt) / (1 - (2*norm_freq/N_filt)**2 + 1e-12)
# Exact rectangular + Hann convolution → use numpy signal
fft_win = np.hanning(N_filt)
fft_win /= fft_win.sum()

# Build prototype PFB filter (Hann-windowed sinc, 4 taps)
M   = n_taps * N_filt
t   = np.arange(M) - M // 2
proto = np.sinc(t / N_filt) * np.hanning(M)
proto /= proto.sum()

# Frequency responses via FFT
nfft = 2**18
W_fft = np.abs(np.fft.fft(fft_win, nfft))
W_pfb = np.abs(np.fft.fft(proto,   nfft * n_taps)[:nfft])
# Normalise and convert to dB
W_fft_db = 20 * np.log10(np.maximum(W_fft / W_fft.max(), 1e-8))
W_pfb_db = 20 * np.log10(np.maximum(W_pfb / W_pfb.max(), 1e-8))
# Shift to centre
bins = np.fft.fftfreq(nfft) * N_filt
bins = np.fft.fftshift(bins)
W_fft_db = np.fft.fftshift(W_fft_db)
W_pfb_db = np.fft.fftshift(W_pfb_db)
xlim = 4.0
sel  = np.abs(bins) <= xlim

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(bins[sel], W_fft_db[sel], color=COLOURS['fft'], label='FFT (Hann)', lw=1.5)
ax.plot(bins[sel], W_pfb_db[sel], color=COLOURS['pfb'], label='PFB (Hann-sinc, 4 taps)', lw=1.5)
ax.axhline(-13.3, color=COLOURS['fft'], ls=':', lw=0.8, alpha=0.6)
ax.axhline(-57.3, color=COLOURS['pfb'], ls=':', lw=0.8, alpha=0.6)
ax.axvline(0.5, color='grey', ls='--', lw=0.7, alpha=0.5)
ax.axvline(-0.5, color='grey', ls='--', lw=0.7, alpha=0.5)
ax.set_xlabel('Frequency offset (bins)')
ax.set_ylabel('Channel response (dB)')
ax.set_title('Fig 1 — Theoretical FFT vs PFB Channel Frequency Response')
ax.set_xlim(-xlim, xlim)
ax.set_ylim(-80, 5)
ax.legend()
# Annotate stopband levels
ax.annotate('FFT first sidelobe\n≈ −13 dB', xy=(1.5, -13.3),
            xytext=(2.2, -5), fontsize=8, color=COLOURS['fft'],
            arrowprops=dict(arrowstyle='->', color=COLOURS['fft'], lw=0.8))
ax.annotate('PFB sidelobe\n≈ −57 dB', xy=(1.5, -57.3),
            xytext=(2.2, -45), fontsize=8, color=COLOURS['pfb'],
            arrowprops=dict(arrowstyle='->', color=COLOURS['pfb'], lw=0.8))
fig.tight_layout()
save_fig(fig, 'fig1_filter_response.png')
print('PASS Fig 1 saved')

Generating Fig 1: Filter response...
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig1_filter_response.png
PASS Fig 1 saved


In [8]:
# CELL 7 - FIG 2: Synthetic EoR detection test (CORRECTED)
# RFI tone placed at HALF-BIN offset from 78 MHz.
# Half-bin offset is worst-case for spectral leakage and is exactly
# the scenario where PFB outperforms FFT in adjacent-bin containment.
print('Generating Fig 2: Synthetic EoR detection test (corrected)...')

rng = np.random.default_rng(42)
N   = N_FFT_COARSE
noise_sigma = 100.0
noise = rng.normal(0, noise_sigma, N).astype(np.float32)

# Half-bin offset from 78 MHz bin
bin_78       = int(EDGES_MHZ / (FS_MHZ / N_FFT_COARSE))
rfi_freq_mhz = (bin_78 + 0.5) * (FS_MHZ / N_FFT_COARSE)
rfi_amp      = 40.0 * noise_sigma
print('  RFI at %.4f MHz (half-bin offset) | bin width %.3f kHz' % (rfi_freq_mhz, DF_COARSE_KHZ))

t_samples = np.arange(N) / (FS_MHZ * 1e6)
rfi_tone  = rfi_amp * np.sin(2 * np.pi * rfi_freq_mhz * 1e6 * t_samples)
signal    = noise + rfi_tone.astype(np.float32)

# FFT spectrometer
win = np.hanning(N).astype(np.float32)
win /= np.sqrt(np.mean(win**2))
fft_out = np.abs(np.fft.rfft(signal * win))**2
fft_db  = 10 * np.log10(np.maximum(fft_out, 1e-10))

# PFB spectrometer (4 taps, Hann-windowed sinc)
M_p     = N_TAPS * N
t_p     = np.arange(M_p) - M_p // 2
proto_p = (np.sinc(t_p / N) * np.hanning(M_p)).astype(np.float32)
proto_p /= np.sqrt(np.mean(proto_p**2))
sig_ext = np.zeros(M_p, dtype=np.float32)
sig_ext[-N:] = signal
pfb_blocks = sig_ext.reshape(N_TAPS, N)
pfb_filt   = np.zeros(N, dtype=np.float32)
for tap_i in range(N_TAPS):
    pfb_filt += pfb_blocks[tap_i] * proto_p[tap_i*N:(tap_i+1)*N]
pfb_out = np.abs(np.fft.rfft(pfb_filt))**2
pfb_db  = 10 * np.log10(np.maximum(pfb_out, 1e-10))

# Extract RHINO band and compute metrics
fft_rhino = fft_db[mask_c]
pfb_rhino = pfb_db[mask_c]
fft_floor = np.median(fft_db[ref_c])
pfb_floor = np.median(pfb_db[ref_c])
rfi_bin   = int(np.argmin(np.abs(f_rhino_c - rfi_freq_mhz)))
fft_snr   = fft_rhino[rfi_bin] - fft_floor
pfb_snr   = pfb_rhino[rfi_bin] - pfb_floor
fft_leak1 = fft_rhino[rfi_bin+1] - fft_floor
pfb_leak1 = pfb_rhino[rfi_bin+1] - pfb_floor
fft_leak2 = fft_rhino[rfi_bin+2] - fft_floor
pfb_leak2 = pfb_rhino[rfi_bin+2] - pfb_floor

# Plot
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
specs  = [fft_rhino, pfb_rhino]
labels = ['FFT (Hann)', 'PFB (Hann-sinc, 4 taps)']
cols   = [COLOURS['fft'], COLOURS['pfb']]
floors = [fft_floor, pfb_floor]
snrs   = [fft_snr, pfb_snr]
for ax, spec, label, col, floor, snr in zip(axes, specs, labels, cols, floors, snrs):
    ax.plot(f_rhino_c, spec, color=col, lw=1.2, label=label)
    ax.axhline(floor, color=col, ls=':', lw=1.0, alpha=0.6, label='Noise floor (60-75 MHz)')
    ax.axvline(EDGES_MHZ, color='k', ls='--', lw=1.0, alpha=0.6, label='EDGES 78 MHz')
    ax.axvline(rfi_freq_mhz, color='r', ls=':', lw=1.0, alpha=0.8,
               label='RFI %.3f MHz (half-bin)' % rfi_freq_mhz)
    ax.annotate('Peak SNR = %.1f dB' % snr,
                xy=(rfi_freq_mhz, spec[rfi_bin]),
                xytext=(rfi_freq_mhz - 5.0, spec[rfi_bin] - 8.0),
                fontsize=9, color=col,
                arrowprops=dict(arrowstyle='->', color=col, lw=0.8))
    ax.set_ylabel('Power (dB, arb.)')
    ax.legend(loc='upper left', fontsize=8)
    ax.set_title(label)
axes[-1].set_xlabel('Frequency (MHz)')
fig.suptitle(
    'Fig 2 - Synthetic EoR Detection Test in RHINO Band (60-85 MHz)\n'
    'RFI at %.3f MHz (half-bin offset, 40x noise) | '
    'FFT peak SNR = %.1f dB | PFB peak SNR = %.1f dB\n'
    'FFT +1bin leakage = %.1f dB | PFB +1bin leakage = %.1f dB | PFB advantage = +%.1f dB'
    % (rfi_freq_mhz, fft_snr, pfb_snr, fft_leak1, pfb_leak1, fft_leak1 - pfb_leak1),
    fontsize=9)
fig.tight_layout()
save_fig(fig, 'fig2_synthetic_eor_detection.png')

print('  RFI freq          : %.4f MHz' % rfi_freq_mhz)
print('  FFT peak SNR      : %.1f dB' % fft_snr)
print('  PFB peak SNR      : %.1f dB' % pfb_snr)
print('  FFT +1 bin leakage: %.1f dB' % fft_leak1)
print('  PFB +1 bin leakage: %.1f dB' % pfb_leak1)
print('  PFB advantage +1  : +%.1f dB' % (fft_leak1 - pfb_leak1))
print('  FFT +2 bin leakage: %.1f dB' % fft_leak2)
print('  PFB +2 bin leakage: %.1f dB' % pfb_leak2)
print('  PFB advantage +2  : +%.1f dB' % (fft_leak2 - pfb_leak2))
print('PASS Fig 2 saved')


Generating Fig 2: Synthetic EoR detection test (corrected)...
  RFI at 77.8950 MHz (half-bin offset) | bin width 270.000 kHz
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig2_synthetic_eor_detection.png
  RFI freq          : 77.8950 MHz
  FFT peak SNR      : 65.9 dB
  PFB peak SNR      : 62.2 dB
  FFT +1 bin leakage: 51.9 dB
  PFB +1 bin leakage: 51.7 dB
  PFB advantage +1  : +0.2 dB
  FFT +2 bin leakage: 35.1 dB
  PFB +2 bin leakage: 41.9 dB
  PFB advantage +2  : +-6.8 dB
PASS Fig 2 saved


In [9]:
# ═══════════════════════════════════════════════════════════════════
# CELL 8 — FIG 3: Wideband spectrum 0–500 MHz, RHINO band highlighted
# Uses DS2 and DS3 full-spectrum coarse arrays.
# ═══════════════════════════════════════════════════════════════════
print('Generating Fig 3: Wideband spectra...')

wideband_limit_mhz = 500.0   # plot 0–500 MHz to keep RHINO band visible
wb_mask = freq_c <= wideband_limit_mhz
f_wb = freq_c[wb_mask]

datasets_wb = []
if ds2.get('fft_coarse') is not None and ds2.get('pfb_coarse') is not None:
    fc2 = get_freq(ds2, 'freq_coarse', freq_c)
    wb2 = (fc2 <= wideband_limit_mhz)
    datasets_wb.append({
        'label'  : 'DS2: Discone + ZKL-2+ LNA',
        'freq'   : fc2[wb2],
        'fft'    : ds2['fft_coarse'][wb2],
        'pfb'    : ds2['pfb_coarse'][wb2],
        'col'    : COLOURS['ds2'],
        'tag'    : 'ds2',
    })

if ds3.get('fft_coarse') is not None and ds3.get('pfb_coarse') is not None:
    fc3 = get_freq(ds3, 'freq_coarse', freq_c)
    wb3 = (fc3 <= wideband_limit_mhz)
    datasets_wb.append({
        'label'  : 'DS3: Yagi + CobraX LNA',
        'freq'   : fc3[wb3],
        'fft'    : ds3['fft_coarse'][wb3],
        'pfb'    : ds3['pfb_coarse'][wb3],
        'col'    : COLOURS['ds3'],
        'tag'    : 'ds3',
    })

if ds4.get('fft_coarse') is not None and ds4.get('pfb_coarse') is not None:
    fc4 = get_freq(ds4, 'freq_coarse', freq_c)
    wb4 = (fc4 <= wideband_limit_mhz)
    datasets_wb.append({
        'label'  : 'DS4: Load + ZKL-2+ LNA (calibration)',
        'freq'   : fc4[wb4],
        'fft'    : ds4['fft_coarse'][wb4],
        'pfb'    : ds4['pfb_coarse'][wb4],
        'col'    : COLOURS['ds4'],
        'tag'    : 'ds4',
    })

if not datasets_wb:
    print('SKIP Fig 3 — no full-spectrum coarse arrays found')
else:
    n_ds = len(datasets_wb)
    fig, axes = plt.subplots(n_ds, 1, figsize=(11, 3.5 * n_ds), squeeze=False)
    for i, ds in enumerate(datasets_wb):
        ax = axes[i][0]
        ax.plot(ds['freq'], ds['fft'], color=COLOURS['fft'], lw=0.8,
                alpha=0.85, label='FFT (Hann)')
        ax.plot(ds['freq'], ds['pfb'], color=COLOURS['pfb'], lw=0.8,
                alpha=0.85, label='PFB (4 taps)')
        # Shade RHINO band
        ax.axvspan(RHINO_LO_MHZ, RHINO_HI_MHZ, alpha=0.12, color='gold',
                   label='RHINO band (60–85 MHz)')
        # Shade FM band
        ax.axvspan(FM_LO_MHZ, FM_HI_MHZ, alpha=0.08, color='red',
                   label='FM (87.5–108 MHz)')
        ax.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, alpha=0.7,
                   label='EDGES 78 MHz (z=17.2)')
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dB, arb.)')
        ax.set_title(ds['label'])
        ax.legend(loc='upper right', fontsize=8)
        ax.set_xlim(0, wideband_limit_mhz)
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
    fig.suptitle('Fig 3 — Wideband Spectrum 0–500 MHz | RHINO Science Band Highlighted',
                 fontsize=11)
    fig.tight_layout()
    save_fig(fig, 'fig3_wideband_spectrum.png')
    print('PASS Fig 3 saved')

Generating Fig 3: Wideband spectra...
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig3_wideband_spectrum.png
PASS Fig 3 saved


In [10]:
# ═══════════════════════════════════════════════════════════════════
# CELL 9 — FIG 4: RHINO band zoom 60–85 MHz, FFT vs PFB
# With dual x-axis (frequency + redshift) and EDGES marker.
# ═══════════════════════════════════════════════════════════════════
print('Generating Fig 4: RHINO band zoom 60–85 MHz...')

datasets_zoom = []
for tag, ds_dict, label in [
        ('ds2', ds2, 'Discone + ZKL-2+ LNA (DS2)'),
        ('ds3', ds3, 'Yagi + CobraX LNA (DS3)'),
        ('ds4', ds4, 'Load + ZKL-2+ LNA — calibration (DS4)')]:
    if ds_dict.get('fft_coarse') is not None and ds_dict.get('pfb_coarse') is not None:
        fc = get_freq(ds_dict, 'freq_coarse', freq_c)
        mk = rhino_mask(fc)
        fft_z = ds_dict['fft_coarse'][mk]
        pfb_z = ds_dict['pfb_coarse'][mk]
        fft_std = float(np.std(fft_z))
        pfb_std = float(np.std(pfb_z))
        datasets_zoom.append({
            'label'  : label,
            'freq'   : fc[mk],
            'fft'    : fft_z,
            'pfb'    : pfb_z,
            'fft_std': fft_std,
            'pfb_std': pfb_std,
            'delta'  : pfb_std - fft_std,
        })
        print('  %-40s FFT std=%.4f dB  PFB std=%.4f dB  Δ=%.4f dB'
              % (label, fft_std, pfb_std, pfb_std - fft_std))

if not datasets_zoom:
    print('SKIP Fig 4 — no coarse arrays with RHINO band data found')
else:
    n_ds = len(datasets_zoom)
    fig, axes = plt.subplots(n_ds, 1, figsize=(10, 4 * n_ds), squeeze=False)
    for i, ds in enumerate(datasets_zoom):
        ax  = axes[i][0]
        ax2 = ax.twiny()   # top axis for redshift
        ax.plot(ds['freq'], ds['fft'], color=COLOURS['fft'], lw=1.2,
                label='FFT (Hann)  std=%.3f dB' % ds['fft_std'])
        ax.plot(ds['freq'], ds['pfb'], color=COLOURS['pfb'], lw=1.2,
                label='PFB (4 taps)  std=%.3f dB' % ds['pfb_std'])
        ax.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.2,
                   label='EDGES (78 MHz, z=17.2)')
        ax.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dB, arb.)')
        ax.set_title(ds['label'] +
                     '  |  PFB − FFT std = %.4f dB' % ds['delta'])
        ax.legend(loc='upper right', fontsize=8)
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
        # Redshift top axis
        f_ticks = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, 6)
        ax2.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
        ax2.set_xticks(f_ticks)
        ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
        ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)

    fig.suptitle('Fig 4 — RHINO Science Band Zoom: 60–85 MHz  |  FFT vs PFB',
                 fontsize=11)
    fig.tight_layout()
    save_fig(fig, 'fig4_rhino_band_zoom.png')
    print('PASS Fig 4 saved')

Generating Fig 4: RHINO band zoom 60–85 MHz...
  Discone + ZKL-2+ LNA (DS2)               FFT std=2.9346 dB  PFB std=3.3588 dB  Δ=0.4242 dB
  Yagi + CobraX LNA (DS3)                  FFT std=1.6159 dB  PFB std=1.8483 dB  Δ=0.2325 dB
  Load + ZKL-2+ LNA — calibration (DS4)    FFT std=0.3406 dB  PFB std=0.3762 dB  Δ=0.0356 dB
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig4_rhino_band_zoom.png
PASS Fig 4 saved


In [11]:
# ═══════════════════════════════════════════════════════════════════
# CELL 10 — FIG 5: Waterfall notice
# The existing waterfall arrays are locked to 1400–1440 MHz.
# New observations required.
# ═══════════════════════════════════════════════════════════════════
print('Fig 5 (FFT vs PFB waterfall at 60–85 MHz):')
print('  WARN All existing waterfall .npy files cover 1400–1440 MHz only.')
print('  WARN New observations required with the notebook corrected to')
print('  WARN RHINO_LO_MHZ=60, RHINO_HI_MHZ=85 before the waterfall cells.')
print('  SKIP Fig 5 cannot be produced from existing data.')

# Produce a placeholder figure so the figure numbering is consistent
fig, ax = plt.subplots(figsize=(8, 3))
ax.text(0.5, 0.5,
    'Fig 5 — FFT vs PFB Waterfall at 60–85 MHz\n\n'
    'NOT AVAILABLE FROM EXISTING DATA\n\n'
    'Existing waterfall arrays cover 1400–1440 MHz only.\n'
    'New observations required: set RHINO_LO_MHZ=60, RHINO_HI_MHZ=85\n'
    'in the acquisition notebook before running waterfall cells.',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
    transform=ax.transAxes)
ax.axis('off')
fig.tight_layout()
save_fig(fig, 'fig5_waterfall_PLACEHOLDER.png')
print('  Placeholder saved.')

Fig 5 (FFT vs PFB waterfall at 60–85 MHz):
  WARN All existing waterfall .npy files cover 1400–1440 MHz only.
  WARN New observations required with the notebook corrected to
  WARN RHINO_LO_MHZ=60, RHINO_HI_MHZ=85 before the waterfall cells.
  SKIP Fig 5 cannot be produced from existing data.
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig5_waterfall_PLACEHOLDER.png
  Placeholder saved.


In [12]:
# ═══════════════════════════════════════════════════════════════════
# CELL 11 — FIG 6: Hi-res spectrum at 60–85 MHz (4.22 kHz/bin)
# Uses DS2 and DS3 hi-res averaged arrays.
# ═══════════════════════════════════════════════════════════════════
print('Generating Fig 6: Hi-res spectrum 60–85 MHz...')

hires_datasets = []
for tag, ds_dict, label in [
        ('ds2', ds2, 'DS2: Discone + ZKL-2+ LNA'),
        ('ds3', ds3, 'DS3: Yagi + CobraX LNA')]:
    fh = get_freq(ds_dict, 'freq_hires', freq_h)
    # Check integration snap if hires avg not saved
    spec_h = ds_dict.get('fft_hires')
    if spec_h is None and ds_dict.get('integ_final') is not None:
        spec_h = ds_dict['integ_final']
        fi = get_freq(ds_dict, 'integ_freq', freq_h)
        fh = fi
    if spec_h is not None and len(spec_h) == len(fh):
        mk_h = rhino_mask(fh)
        hires_datasets.append({
            'label': label,
            'freq' : fh[mk_h],
            'spec' : spec_h[mk_h],
        })

if not hires_datasets:
    print('SKIP Fig 6 — no hi-res arrays found')
else:
    n_ds = len(hires_datasets)
    fig, axes = plt.subplots(n_ds, 1, figsize=(11, 4 * n_ds), squeeze=False)
    for i, ds in enumerate(hires_datasets):
        ax  = axes[i][0]
        ax2 = ax.twiny()
        ax.plot(ds['freq'], ds['spec'], color=COLOURS['fft'], lw=0.6,
                alpha=0.85, label='Hi-res FFT (%.3f kHz/bin)' % DF_HIRES_KHZ)
        ax.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.2,
                   label='EDGES (78 MHz, z=17.2)')
        ax.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dB, arb.)')
        ax.set_title(ds['label'])
        ax.legend(loc='upper right', fontsize=8)
        ax.xaxis.set_minor_locator(ticker.MultipleLocator(1))
        # Redshift top axis
        f_ticks = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, 6)
        ax2.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
        ax2.set_xticks(f_ticks)
        ax2.set_xticklabels(['z=%.1f' % freq_to_z(f) for f in f_ticks], fontsize=8)
        ax2.set_xlabel('Redshift  z = ν₂₁/ν − 1', fontsize=9)
    fig.suptitle('Fig 6 — Hi-res Spectrum at 60–85 MHz  |  %.3f kHz/bin  |  N=1,048,576'
                 % DF_HIRES_KHZ, fontsize=11)
    fig.tight_layout()
    save_fig(fig, 'fig6_hires_spectrum_60_85MHz.png')
    print('PASS Fig 6 saved')

Generating Fig 6: Hi-res spectrum 60–85 MHz...
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig6_hires_spectrum_60_85MHz.png
PASS Fig 6 saved


In [13]:
# ═══════════════════════════════════════════════════════════════════
# CELL 12 — FIG 7: Hi-res waterfall notice (same as Fig 5)
# ═══════════════════════════════════════════════════════════════════
print('Fig 7 (Hi-res waterfall at 60–85 MHz):')
print('  SKIP Same situation as Fig 5 — waterfall arrays locked to 1400–1440 MHz.')
fig, ax = plt.subplots(figsize=(8, 3))
ax.text(0.5, 0.5,
    'Fig 7 — Hi-res Waterfall at 60–85 MHz\n\n'
    'NOT AVAILABLE FROM EXISTING DATA\n\n'
    'Hi-res waterfall arrays cover 1400–1440 MHz only.\n'
    'New observations required with the corrected acquisition notebook.',
    ha='center', va='center', fontsize=11,
    bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8),
    transform=ax.transAxes)
ax.axis('off')
fig.tight_layout()
save_fig(fig, 'fig7_hires_waterfall_PLACEHOLDER.png')
print('  Placeholder saved.')

Fig 7 (Hi-res waterfall at 60–85 MHz):
  SKIP Same situation as Fig 5 — waterfall arrays locked to 1400–1440 MHz.
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig7_hires_waterfall_PLACEHOLDER.png
  Placeholder saved.


In [14]:
# ═══════════════════════════════════════════════════════════════════
# CELL 13 — FIG 8: Radiometer equation — noise vs integration depth
# Uses integration snapshots from DS1, DS2, DS3, DS4 and
# DS4-Long checkpoints.
# Reference sub-band: 60–75 MHz (RFI-quiet within RHINO band).
# ═══════════════════════════════════════════════════════════════════
print('Generating Fig 8: Radiometer equation...')

def build_radiometer_curve(snaps_raw, freq_arr):
    """
    Build radiometer curve from a list of (filename, array) snapshot tuples.
    Each array is a full-spectrum hi-res or coarse spectrum in dB.
    Returns (N_values, std_values) arrays sorted by N.
    """
    rm = ref_mask(freq_arr)
    if rm.sum() == 0:
        print('  WARN ref mask empty for this freq axis')
        return np.array([]), np.array([])
    points = []
    for fname, arr in snaps_raw:
        if len(arr) != len(freq_arr):
            continue
        N  = extract_N(fname)
        std = float(np.std(arr[rm]))
        if np.isfinite(std) and std > 0:
            points.append((N, std))
    if not points:
        return np.array([]), np.array([])
    points.sort()
    Ns   = np.array([p[0] for p in points], dtype=float)
    stds = np.array([p[1] for p in points], dtype=float)
    return Ns, stds

def build_checkpoint_curve(checkpoints, freq_arr):
    """
    Build radiometer curve from checkpoint arrays (coarse, 8193 bins).
    """
    rm = ref_mask(freq_arr)
    if rm.sum() == 0:
        return np.array([]), np.array([])
    points = []
    for fname, arr in checkpoints:
        if len(arr) != len(freq_arr):
            continue
        N   = extract_N(fname)
        std = float(np.std(arr[rm]))
        if np.isfinite(std) and std > 0:
            points.append((N, std))
    if not points:
        return np.array([]), np.array([])
    points.sort()
    return np.array([p[0] for p in points]), np.array([p[1] for p in points])

# ── Collect curves ───────────────────────────────────────────────
curves = []

# DS1 — hi-res snaps
if ds1.get('snaps_raw') and ds1.get('integ_freq') is not None:
    fi1 = get_freq(ds1, 'integ_freq', freq_h)
    Ns1, std1 = build_radiometer_curve(ds1['snaps_raw'], fi1)
    if len(Ns1) > 2:
        curves.append({'label': 'DS1: Discone, no LNA', 'N': Ns1, 'std': std1,
                       'col': COLOURS['ds1']})
        print('  DS1: %d points  ratio=%.2f' %
              (len(Ns1), std1[0] * np.sqrt(Ns1[0]) / (std1[-1] * np.sqrt(Ns1[-1]))))

# DS2 — hi-res snaps
if ds2.get('snaps_raw') and ds2.get('integ_freq') is not None:
    fi2 = get_freq(ds2, 'integ_freq', freq_h)
    Ns2, std2 = build_radiometer_curve(ds2['snaps_raw'], fi2)
    if len(Ns2) > 2:
        curves.append({'label': 'DS2: Discone + ZKL-2+ LNA', 'N': Ns2, 'std': std2,
                       'col': COLOURS['ds2']})
        print('  DS2: %d points  ratio=%.2f' %
              (len(Ns2), std2[0] * np.sqrt(Ns2[0]) / (std2[-1] * np.sqrt(Ns2[-1]))))

# DS3 — hi-res snaps
if ds3.get('snaps_raw') and ds3.get('integ_freq') is not None:
    fi3 = get_freq(ds3, 'integ_freq', freq_h)
    Ns3, std3 = build_radiometer_curve(ds3['snaps_raw'], fi3)
    if len(Ns3) > 2:
        curves.append({'label': 'DS3: Yagi + CobraX LNA', 'N': Ns3, 'std': std3,
                       'col': COLOURS['ds3']})
        print('  DS3: %d points  ratio=%.2f' %
              (len(Ns3), std3[0] * np.sqrt(Ns3[0]) / (std3[-1] * np.sqrt(Ns3[-1]))))

# DS4 — hi-res snaps
if ds4.get('snaps_raw') and ds4.get('integ_freq') is not None:
    fi4 = get_freq(ds4, 'integ_freq', freq_h)
    Ns4, std4 = build_radiometer_curve(ds4['snaps_raw'], fi4)
    if len(Ns4) > 2:
        curves.append({'label': 'DS4: Load + ZKL-2+ LNA (~21 min)', 'N': Ns4, 'std': std4,
                       'col': COLOURS['ds4']})
        print('  DS4: %d points  ratio=%.2f' %
              (len(Ns4), std4[0] * np.sqrt(Ns4[0]) / (std4[-1] * np.sqrt(Ns4[-1]))))

# DS4-Long — coarse checkpoints
if ds4l.get('checkpoints') and ds4l.get('freq_coarse') is not None:
    fL = ds4l['freq_coarse']
    NsL, stdL = build_checkpoint_curve(ds4l['checkpoints'], fL)
    if len(NsL) > 2:
        curves.append({'label': 'DS4-Long: Load + ZKL-2+ LNA (3 h)', 'N': NsL, 'std': stdL,
                       'col': COLOURS['ds4l']})
        print('  DS4-Long: %d points  ratio=%.2f' %
              (len(NsL), stdL[0] * np.sqrt(NsL[0]) / (stdL[-1] * np.sqrt(NsL[-1]))))

if not curves:
    print('SKIP Fig 8 — no integration snapshots found')
else:
    fig, ax = plt.subplots(figsize=(9, 5))
    # Plot ideal 1/sqrt(N) for reference
    N_ideal = np.geomspace(1, max(c['N'].max() for c in curves), 200)
    # Scale ideal to first point of first curve
    c0 = curves[0]
    scale = c0['std'][0] * np.sqrt(c0['N'][0])
    ax.plot(N_ideal, scale / np.sqrt(N_ideal), 'k--', lw=1.5, alpha=0.5,
            label='Ideal 1/√N  (radiometer equation)')
    for c in curves:
        ax.scatter(c['N'], c['std'], s=18, color=c['col'], alpha=0.8, zorder=5)
        ax.plot(c['N'], c['std'], color=c['col'], lw=1.0, alpha=0.7,
                label=c['label'])
        # Compute and annotate compliance ratio
        ratio = (c['std'][0] * np.sqrt(c['N'][0])) / (c['std'][-1] * np.sqrt(c['N'][-1]))
        ax.annotate('ratio=%.2f' % ratio,
                    xy=(c['N'][-1], c['std'][-1]),
                    xytext=(c['N'][-1] * 0.7, c['std'][-1] * 1.3),
                    fontsize=7, color=c['col'],
                    arrowprops=dict(arrowstyle='->', color=c['col'], lw=0.6))
    ax.set_xscale('log'); ax.set_yscale('log')
    ax.set_xlabel('Number of averaged spectra  N')
    ax.set_ylabel('Spectral std dev in ref band 60–75 MHz  (dB)')
    ax.set_title(
        'Fig 8 — Radiometer Equation: Noise vs Integration Depth\n'
        'Reference sub-band: 60–75 MHz  (RFI-quiet zone within RHINO band)')
    ax.legend(fontsize=8)
    fig.tight_layout()
    save_fig(fig, 'fig8_radiometer_equation.png')
    print('PASS Fig 8 saved')

Generating Fig 8: Radiometer equation...
  DS1: 10 points  ratio=0.37
  DS2: 30 points  ratio=0.19
  DS3: 30 points  ratio=0.27
  DS4: 30 points  ratio=0.95
  DS4-Long: 45 points  ratio=0.96
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig8_radiometer_equation.png
PASS Fig 8 saved


In [15]:
# ═══════════════════════════════════════════════════════════════════
# CELL 14 — BONUS: DS1 radiometer from hi-res integration snaps
# DS1 had no full-spectrum coarse arrays, but the integration snaps
# are hi-res full-spectrum — extract 60–85 MHz noise evolution.
# ═══════════════════════════════════════════════════════════════════
print('Generating Bonus: DS1 integration depth comparison at 60–85 MHz...')

if ds1.get('snaps_raw') and ds1.get('integ_freq') is not None:
    fi1 = get_freq(ds1, 'integ_freq', freq_h)
    mk1 = rhino_mask(fi1)
    if mk1.sum() > 0:
        valid_snaps = [(n, a) for (n, a) in ds1['snaps_raw'] if len(a) == len(fi1)]
        if len(valid_snaps) >= 3:
            # Show spectrum evolution: first, middle, last snap
            idx_show = [0, len(valid_snaps)//2, len(valid_snaps)-1]
            fig, ax = plt.subplots(figsize=(10, 4))
            cmap = plt.cm.Blues
            for j, idx in enumerate(idx_show):
                fname, arr = valid_snaps[idx]
                N_val = extract_N(fname)
                col   = cmap(0.4 + 0.5 * j / max(len(idx_show)-1, 1))
                ax.plot(fi1[mk1], arr[mk1], color=col, lw=0.9, alpha=0.9,
                        label='N=%d' % N_val)
            ax.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, label='EDGES 78 MHz')
            ax.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
            ax.set_xlabel('Frequency (MHz)')
            ax.set_ylabel('Power (dB, arb.)')
            ax.set_title('DS1 (Discone, no LNA) — Integration Depth Comparison at 60–85 MHz')
            ax.legend(fontsize=9)
            fig.tight_layout()
            save_fig(fig, 'bonus_ds1_integration_depth_comparison.png')
            print('PASS Bonus DS1 saved')
        else:
            print('SKIP DS1 — fewer than 3 valid hi-res snapshots')
    else:
        print('SKIP DS1 — frequency mask covers 0 bins')
else:
    print('SKIP DS1 — no hi-res integration snaps loaded')

Generating Bonus: DS1 integration depth comparison at 60–85 MHz...
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/bonus_ds1_integration_depth_comparison.png
PASS Bonus DS1 saved


In [16]:
# ═══════════════════════════════════════════════════════════════════
# CELL 15 — BONUS: DS4-Long — 3-hour receiver stability at 60–85 MHz
# Shows the thermal noise floor evolving over 2266 spectra.
# ═══════════════════════════════════════════════════════════════════
print('Generating Bonus: DS4-Long 3-hour receiver stability...')

if ds4l.get('checkpoints') and ds4l.get('freq_coarse') is not None:
    fL   = ds4l['freq_coarse']
    mkL  = rhino_mask(fL)
    if mkL.sum() > 0 and len(ds4l['checkpoints']) >= 3:
        # Show spectrum at start, middle, and end of 3-hour run
        cps  = ds4l['checkpoints']
        idx_show = [0, len(cps)//2, len(cps)-1]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Left: spectral comparison at 3 depths
        ax = axes[0]
        cmap = plt.cm.Oranges
        for j, idx in enumerate(idx_show):
            fname, arr = cps[idx]
            N_val = extract_N(fname)
            col   = cmap(0.4 + 0.5 * j / max(len(idx_show)-1, 1))
            ax.plot(fL[mkL], arr[mkL], color=col, lw=0.9,
                    label='N=%d' % N_val)
        ax.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, label='EDGES 78 MHz')
        ax.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
        ax.set_xlabel('Frequency (MHz)')
        ax.set_ylabel('Power (dB, arb.)')
        ax.set_title('Load + LNA Spectrum at 60–85 MHz\n(3-hour integration, coarse)')
        ax.legend(fontsize=8)

        # Right: std dev in reference band vs N (radiometer curve)
        ax2 = axes[1]
        Ns_L = []
        std_L = []
        ref_L = ref_mask(fL)
        for fname, arr in cps:
            if len(arr) == len(fL) and ref_L.sum() > 0:
                N_val = extract_N(fname)
                std   = float(np.std(arr[ref_L]))
                if np.isfinite(std) and std > 0:
                    Ns_L.append(N_val)
                    std_L.append(std)
        Ns_L  = np.array(Ns_L)
        std_L = np.array(std_L)
        if len(Ns_L) > 2:
            scale = std_L[0] * np.sqrt(Ns_L[0])
            N_id  = np.geomspace(Ns_L.min(), Ns_L.max(), 100)
            ax2.plot(N_id, scale / np.sqrt(N_id), 'k--', lw=1.5,
                     alpha=0.5, label='Ideal 1/√N')
            ax2.plot(Ns_L, std_L, color=COLOURS['ds4l'], lw=1.2,
                     label='Load+LNA (3 h)')
            ax2.scatter(Ns_L, std_L, s=12, color=COLOURS['ds4l'], zorder=5)
            ratio = (std_L[0] * np.sqrt(Ns_L[0])) / (std_L[-1] * np.sqrt(Ns_L[-1]))
            ax2.set_xscale('log'); ax2.set_yscale('log')
            ax2.set_xlabel('N (averaged spectra)')
            ax2.set_ylabel('Std dev in 60–75 MHz (dB)')
            ax2.set_title('3-Hour Radiometer Curve\ncompliance ratio = %.2f' % ratio)
            ax2.legend(fontsize=8)

        fig.suptitle('Bonus — DS4-Long: 3-Hour Receiver Stability at 60–85 MHz\n'
                     'Load + ZKL-2+ LNA | 2266 spectra | Coarse 270 kHz/bin',
                     fontsize=10)
        fig.tight_layout()
        save_fig(fig, 'bonus_ds4long_3hour_stability.png')
        print('PASS Bonus DS4-Long saved')
    else:
        print('SKIP DS4-Long — insufficient data')
else:
    print('SKIP DS4-Long — checkpoints or freq axis not loaded')

Generating Bonus: DS4-Long 3-hour receiver stability...
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/bonus_ds4long_3hour_stability.png
PASS Bonus DS4-Long saved


In [17]:
# ═══════════════════════════════════════════════════════════════════
# CELL 16 — Summary metrics table
# Prints all key numbers for the thesis table.
# ═══════════════════════════════════════════════════════════════════
print('=' * 70)
print('RHINO SCIENCE BAND METRICS SUMMARY  (60–85 MHz)')
print('=' * 70)

print('\n--- FFT vs PFB spectral flatness (coarse, 60–85 MHz) ---')
for tag, ds_dict, label in [
        ('ds2', ds2, 'DS2 Discone+LNA'),
        ('ds3', ds3, 'DS3 Yagi+LNA'),
        ('ds4', ds4, 'DS4 Load+LNA')]:
    if ds_dict.get('fft_coarse') is not None and ds_dict.get('pfb_coarse') is not None:
        fc = get_freq(ds_dict, 'freq_coarse', freq_c)
        mk = rhino_mask(fc)
        if mk.sum() > 0:
            fs = float(np.std(ds_dict['fft_coarse'][mk]))
            ps = float(np.std(ds_dict['pfb_coarse'][mk]))
            print('  %-20s  FFT std = %6.4f dB   PFB std = %6.4f dB   Δ = %+.4f dB'
                  % (label, fs, ps, ps - fs))

print('\n--- Radiometer compliance ratios (60–75 MHz reference band) ---')
for c in curves:
    if len(c['N']) > 1:
        ratio = (c['std'][0] * np.sqrt(c['N'][0])) / (c['std'][-1] * np.sqrt(c['N'][-1]))
        print('  %-38s  N=%4d→%4d   ratio = %.2f' %
              (c['label'], int(c['N'][0]), int(c['N'][-1]), ratio))

print('\n--- Hi-res resolution ---')
print('  N = %d   Δf = %.3f kHz/bin   bins in 60–85 MHz = %d'
      % (N_FFT_HIRES, DF_HIRES_KHZ, mask_h.sum()))
print('  Meets Phil\'s <10 kHz requirement: YES (%.3f kHz/bin)' % DF_HIRES_KHZ)

print('\n--- Frequency / redshift ---')
print('  RHINO band  : %.0f – %.0f MHz  →  z = %.1f – %.1f'
      % (RHINO_LO_MHZ, RHINO_HI_MHZ, z_lo, z_hi))
print('  EDGES (78 MHz) : z = %.2f' % z_edges)
print('  FM broadcast   : %.1f – %.1f MHz (begins %.1f MHz above RHINO)'
      % (FM_LO_MHZ, FM_HI_MHZ, FM_LO_MHZ - RHINO_HI_MHZ))

print('\n--- Output directory ---')
saved = sorted(glob.glob(os.path.join(OUT, '*.png')))
print('  %d figures saved to: %s' % (len(saved), OUT))
for f in saved:
    print('  ', os.path.basename(f))

print('\nPASS all done')

RHINO SCIENCE BAND METRICS SUMMARY  (60–85 MHz)

--- FFT vs PFB spectral flatness (coarse, 60–85 MHz) ---
  DS2 Discone+LNA       FFT std = 2.9346 dB   PFB std = 3.3588 dB   Δ = +0.4242 dB
  DS3 Yagi+LNA          FFT std = 1.6159 dB   PFB std = 1.8483 dB   Δ = +0.2325 dB
  DS4 Load+LNA          FFT std = 0.3406 dB   PFB std = 0.3762 dB   Δ = +0.0356 dB

--- Radiometer compliance ratios (60–75 MHz reference band) ---
  DS1: Discone, no LNA                    N=  12→ 117   ratio = 0.37
  DS2: Discone + ZKL-2+ LNA               N=  12→ 349   ratio = 0.19
  DS3: Yagi + CobraX LNA                  N=  12→ 349   ratio = 0.27
  DS4: Load + ZKL-2+ LNA (~21 min)        N=  12→ 347   ratio = 0.95
  DS4-Long: Load + ZKL-2+ LNA (3 h)       N=  50→2250   ratio = 0.96

--- Hi-res resolution ---
  N = 1048576   Δf = 4.219 kHz/bin   bins in 60–85 MHz = 5926
  Meets Phil's <10 kHz requirement: YES (4.219 kHz/bin)

--- Frequency / redshift ---
  RHINO band  : 60 – 85 MHz  →  z = 15.7 – 22.7
  EDGES (78 

In [19]:
# CELL 17 - FIG 5 & FIG 7: Waterfall figures (post new JBO observations)
# Run this cell AFTER new observations with the acquisition notebook
# corrected to: RHINO_LO_MHZ=60, RHINO_HI_MHZ=85, HI_FREQ_MHZ=78.0
# New waterfall files will have shape (n_frames, ~92) for coarse
# and (n_frames, ~5926) for hi-res, not 148 or 9481 as before.
# Change WATERFALL_PATH if files are in a different directory.
print('Cell 17: Loading new waterfall files...')

WATERFALL_PATH = '/Users/user/Downloads/Manny-Masters/Project/Data/Jodrell_Discone_New/LNA'   # new 22 Jun 2025 discone+LNA run
WATERFALL_PATH_NOLNA = '/Users/user/Downloads/Manny-Masters/Project/Data/Jodrell_Discone_New/No_LNA'  # new 22 Jun 2025 discone no-LNA run
OLD_COLS = [148, 9481]           # old 1400-1440 MHz column counts (reject these)

def find_new_wf(path, prefix):
    import glob as _g
    files = sorted(_g.glob(os.path.join(path, prefix + '*.npy')))
    valid = []
    for fp in files:
        try:
            arr = np.load(fp, allow_pickle=False)
            if arr.ndim == 2 and arr.shape[1] not in OLD_COLS and arr.shape[1] > 0 and arr.shape[1] >= 90:
                valid.append((fp, arr))
                print('  OK  %-50s shape=%s' % (os.path.basename(fp), arr.shape))
            else:
                print('  SKIP%-50s shape=%s (old band)' % (os.path.basename(fp), arr.shape))
        except Exception as ex:
            print('  FAIL%-50s %s' % (os.path.basename(fp), ex))
    return valid[-1] if valid else None

wf_fft_new   = find_new_wf(WATERFALL_PATH, 'wf_fft_')
wf_pfb_new   = find_new_wf(WATERFALL_PATH, 'wf_pfb_')
wf_hires_new = find_new_wf(WATERFALL_PATH, 'wf_hires_')

# FIG 5: coarse FFT vs PFB waterfall
if wf_fft_new is not None and wf_pfb_new is not None:
    _, wf_fft_arr = wf_fft_new
    _, wf_pfb_arr = wf_pfb_new
    n_frames_c, n_bins_c = wf_fft_arr.shape
    # Build freq axis to exactly match the waterfall column count
    # (acquisition notebook may save 92 or 93 bins depending on slice edge)
    f_wf_c = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, n_bins_c)
    wf_diff = wf_fft_arr - wf_pfb_arr
    mean_diff = float(np.mean(np.abs(wf_diff)))

    fig = plt.figure(figsize=(14, 9))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)

    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.imshow(wf_fft_arr, aspect='auto', origin='upper',
                     extent=[f_wf_c[0], f_wf_c[-1], n_frames_c, 0], cmap='viridis')
    ax1.axvline(EDGES_MHZ, color='w', ls='--', lw=0.8, alpha=0.7)
    ax1.set_xlabel('Frequency (MHz)'); ax1.set_ylabel('Frame')
    ax1.set_title('FFT (Hann)')
    plt.colorbar(im1, ax=ax1, label='dB')

    ax2 = fig.add_subplot(gs[0, 1])
    im2 = ax2.imshow(wf_pfb_arr, aspect='auto', origin='upper',
                     extent=[f_wf_c[0], f_wf_c[-1], n_frames_c, 0], cmap='viridis')
    ax2.axvline(EDGES_MHZ, color='w', ls='--', lw=0.8, alpha=0.7)
    ax2.set_xlabel('Frequency (MHz)'); ax2.set_ylabel('Frame')
    ax2.set_title('PFB (4 taps)')
    plt.colorbar(im2, ax=ax2, label='dB')

    ax3 = fig.add_subplot(gs[0, 2])
    vlim = float(np.percentile(np.abs(wf_diff), 95))
    im3 = ax3.imshow(wf_diff, aspect='auto', origin='upper',
                     extent=[f_wf_c[0], f_wf_c[-1], n_frames_c, 0],
                     cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax3.axvline(EDGES_MHZ, color='k', ls='--', lw=0.8, alpha=0.7)
    ax3.set_xlabel('Frequency (MHz)'); ax3.set_ylabel('Frame')
    ax3.set_title('FFT - PFB difference')
    plt.colorbar(im3, ax=ax3, label='dB')

    ax4 = fig.add_subplot(gs[1, :])
    ax4.plot(f_wf_c, np.mean(wf_fft_arr, axis=0), color=COLOURS['fft'], lw=1.2, label='FFT mean')
    ax4.plot(f_wf_c, np.mean(wf_pfb_arr, axis=0), color=COLOURS['pfb'], lw=1.2, label='PFB mean')
    ax4.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, alpha=0.7, label='EDGES 78 MHz (z=17.2)')
    ax4.set_xlabel('Frequency (MHz)'); ax4.set_ylabel('Mean power (dB)')
    ax4.set_title('Time-averaged spectrum (%d frames)' % n_frames_c)
    ax4.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    ax4.legend(fontsize=9)
    ax4.text(0.02, 0.05, 'mean|FFT-PFB| = %.4f dB' % mean_diff,
             transform=ax4.transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))

    fig.suptitle('Fig 5 - FFT vs PFB Waterfall | RHINO Band 60-85 MHz\n'
                 'Discone + ZKL-2+ LNA | %d frames | 270 kHz/bin' % n_frames_c, fontsize=11)
    save_fig(fig, 'fig5_waterfall_60_85MHz.png')
    print('  mean|FFT-PFB| = %.4f dB' % mean_diff)
    print('PASS Fig 5 saved')
else:
    print('SKIP Fig 5 - no new coarse waterfall files found in:', WATERFALL_PATH)
    print('  Complete new JBO observations then re-run this cell.')

# FIG 7: hi-res waterfall
if wf_hires_new is not None:
    _, wf_hires_arr = wf_hires_new
    n_frames_h, n_bins_h = wf_hires_arr.shape
    # Build hi-res freq axis to exactly match waterfall column count
    f_wf_h = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, n_bins_h)

    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    im = axes[0].imshow(wf_hires_arr, aspect='auto', origin='upper',
                        extent=[f_wf_h[0], f_wf_h[-1], n_frames_h, 0], cmap='viridis')
    axes[0].axvline(EDGES_MHZ, color='w', ls='--', lw=1.0, alpha=0.8)
    axes[0].set_xlabel('Frequency (MHz)'); axes[0].set_ylabel('Frame')
    axes[0].set_title('Hi-res FFT Waterfall | %.3f kHz/bin' % DF_HIRES_KHZ)
    plt.colorbar(im, ax=axes[0], label='dB')

    axes[1].plot(f_wf_h, np.mean(wf_hires_arr, axis=0),
                 color=COLOURS['fft'], lw=0.7, alpha=0.8, label='Mean')
    axes[1].plot(f_wf_h, np.max(wf_hires_arr, axis=0),
                 color='red', lw=0.7, alpha=0.6, label='Max-hold')
    axes[1].axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, alpha=0.8,
                    label='EDGES 78 MHz (z=17.2)')
    axes[1].set_xlabel('Frequency (MHz)'); axes[1].set_ylabel('Power (dB)')
    axes[1].set_title('Mean and Max-hold | %d frames' % n_frames_h)
    axes[1].set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    axes[1].legend(fontsize=9)

    fig.suptitle('Fig 7 - Hi-res FFT Waterfall | RHINO Band 60-85 MHz\n'
                 '%.3f kHz/bin | N=1,048,576 | %d frames' % (DF_HIRES_KHZ, n_frames_h),
                 fontsize=11)
    fig.tight_layout()
    save_fig(fig, 'fig7_hires_waterfall_60_85MHz.png')
    print('PASS Fig 7 saved')
else:
    print('SKIP Fig 7 - no new hi-res waterfall files found in:', WATERFALL_PATH)
    print('  Complete new JBO observations then re-run this cell.')

# ── FIG 5b & FIG 7b: No-LNA run waterfalls ──────────────────────
print('\nLoading No-LNA waterfall files...')
wf_fft_nolna   = find_new_wf(WATERFALL_PATH_NOLNA, 'wf_fft_')
wf_pfb_nolna   = find_new_wf(WATERFALL_PATH_NOLNA, 'wf_pfb_')
wf_hires_nolna = find_new_wf(WATERFALL_PATH_NOLNA, 'wf_hires_')

if wf_fft_nolna is not None and wf_pfb_nolna is not None:
    _, wf_fft_arr = wf_fft_nolna
    _, wf_pfb_arr = wf_pfb_nolna
    n_frames_c, n_bins_c = wf_fft_arr.shape
    # Build freq axis to exactly match the waterfall column count
    # (acquisition notebook may save 92 or 93 bins depending on slice edge)
    f_wf_c = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, n_bins_c)
    wf_diff = wf_fft_arr - wf_pfb_arr
    mean_diff = float(np.mean(np.abs(wf_diff)))

    fig = plt.figure(figsize=(14, 9))
    gs  = GridSpec(2, 3, figure=fig, hspace=0.4, wspace=0.35)
    ax1 = fig.add_subplot(gs[0, 0])
    im1 = ax1.imshow(wf_fft_arr, aspect='auto', origin='upper',
                     extent=[f_wf_c[0], f_wf_c[-1], n_frames_c, 0], cmap='viridis')
    ax1.axvline(EDGES_MHZ, color='w', ls='--', lw=0.8, alpha=0.7)
    ax1.set_xlabel('Frequency (MHz)'); ax1.set_ylabel('Frame')
    ax1.set_title('FFT (Hann) — No LNA')
    plt.colorbar(im1, ax=ax1, label='dB')
    ax2 = fig.add_subplot(gs[0, 1])
    im2 = ax2.imshow(wf_pfb_arr, aspect='auto', origin='upper',
                     extent=[f_wf_c[0], f_wf_c[-1], n_frames_c, 0], cmap='viridis')
    ax2.axvline(EDGES_MHZ, color='w', ls='--', lw=0.8, alpha=0.7)
    ax2.set_xlabel('Frequency (MHz)'); ax2.set_ylabel('Frame')
    ax2.set_title('PFB (4 taps) — No LNA')
    plt.colorbar(im2, ax=ax2, label='dB')
    ax3 = fig.add_subplot(gs[0, 2])
    vlim = float(np.percentile(np.abs(wf_diff), 95))
    im3 = ax3.imshow(wf_diff, aspect='auto', origin='upper',
                     extent=[f_wf_c[0], f_wf_c[-1], n_frames_c, 0],
                     cmap='RdBu_r', vmin=-vlim, vmax=vlim)
    ax3.axvline(EDGES_MHZ, color='k', ls='--', lw=0.8, alpha=0.7)
    ax3.set_xlabel('Frequency (MHz)'); ax3.set_ylabel('Frame')
    ax3.set_title('FFT - PFB difference — No LNA')
    plt.colorbar(im3, ax=ax3, label='dB')
    ax4 = fig.add_subplot(gs[1, :])
    ax4.plot(f_wf_c, np.mean(wf_fft_arr, axis=0), color=COLOURS['fft'], lw=1.2, label='FFT mean')
    ax4.plot(f_wf_c, np.mean(wf_pfb_arr, axis=0), color=COLOURS['pfb'], lw=1.2, label='PFB mean')
    ax4.axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, alpha=0.7, label='EDGES 78 MHz (z=17.2)')
    ax4.set_xlabel('Frequency (MHz)'); ax4.set_ylabel('Mean power (dB)')
    ax4.set_title('Time-averaged spectrum (%d frames) — No LNA' % n_frames_c)
    ax4.set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    ax4.legend(fontsize=9)
    ax4.text(0.02, 0.05, 'mean|FFT-PFB| = %.4f dB' % mean_diff,
             transform=ax4.transAxes, fontsize=9,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.7))
    fig.suptitle('Fig 5b - FFT vs PFB Waterfall | RHINO Band 60-85 MHz\n'
                 'Discone, No LNA | %d frames | 270 kHz/bin' % n_frames_c, fontsize=11)
    save_fig(fig, 'fig5b_waterfall_nolna_60_85MHz.png')
    print('  mean|FFT-PFB| No-LNA = %.4f dB' % mean_diff)
    print('PASS Fig 5b saved')
else:
    print('SKIP Fig 5b - no No-LNA coarse waterfall files found')

if wf_hires_nolna is not None:
    _, wf_hires_arr = wf_hires_nolna
    n_frames_h, n_bins_h = wf_hires_arr.shape
    # Build hi-res freq axis to exactly match waterfall column count
    f_wf_h = np.linspace(RHINO_LO_MHZ, RHINO_HI_MHZ, n_bins_h)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8))
    im = axes[0].imshow(wf_hires_arr, aspect='auto', origin='upper',
                        extent=[f_wf_h[0], f_wf_h[-1], n_frames_h, 0], cmap='viridis')
    axes[0].axvline(EDGES_MHZ, color='w', ls='--', lw=1.0, alpha=0.8)
    axes[0].set_xlabel('Frequency (MHz)'); axes[0].set_ylabel('Frame')
    axes[0].set_title('Hi-res FFT Waterfall | %.3f kHz/bin | No LNA' % DF_HIRES_KHZ)
    plt.colorbar(im, ax=axes[0], label='dB')
    axes[1].plot(f_wf_h, np.mean(wf_hires_arr, axis=0),
                 color=COLOURS['fft'], lw=0.7, alpha=0.8, label='Mean')
    axes[1].plot(f_wf_h, np.max(wf_hires_arr, axis=0),
                 color='red', lw=0.7, alpha=0.6, label='Max-hold')
    axes[1].axvline(EDGES_MHZ, color='navy', ls='--', lw=1.0, alpha=0.8,
                    label='EDGES 78 MHz (z=17.2)')
    axes[1].set_xlabel('Frequency (MHz)'); axes[1].set_ylabel('Power (dB)')
    axes[1].set_title('Mean and Max-hold | %d frames | No LNA' % n_frames_h)
    axes[1].set_xlim(RHINO_LO_MHZ, RHINO_HI_MHZ)
    axes[1].legend(fontsize=9)
    fig.suptitle('Fig 7b - Hi-res FFT Waterfall | RHINO Band 60-85 MHz\n'
                 '%.3f kHz/bin | N=1,048,576 | %d frames | No LNA' % (DF_HIRES_KHZ, n_frames_h),
                 fontsize=11)
    fig.tight_layout()
    save_fig(fig, 'fig7b_hires_waterfall_nolna_60_85MHz.png')
    print('PASS Fig 7b saved')
else:
    print('SKIP Fig 7b - no No-LNA hi-res waterfall files found')


Cell 17: Loading new waterfall files...
  OK  wf_fft_20250622_021913.npy                         shape=(120, 93)
  OK  wf_fft_20250622_030852.npy                         shape=(120, 93)
  OK  wf_pfb_20250622_021913.npy                         shape=(120, 93)
  OK  wf_pfb_20250622_030852.npy                         shape=(120, 93)
  OK  wf_hires_20250622_023154.npy                       shape=(40, 5926)
  OK  wf_hires_20250622_030852.npy                       shape=(40, 5926)
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig5_waterfall_60_85MHz.png
  mean|FFT-PFB| = 8.2485 dB
PASS Fig 5 saved
  Saved: /Users/user/Downloads/Manny-Masters/Project/Data/rhino_figures_60_85MHz/fig7_hires_waterfall_60_85MHz.png
PASS Fig 7 saved

Loading No-LNA waterfall files...
  OK  wf_fft_20250622_031418.npy                         shape=(120, 93)
  OK  wf_fft_20250622_040048.npy                         shape=(120, 93)
  OK  wf_pfb_20250622_031418.npy                     

In [20]:
import numpy as np

BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'
P5   = BASE + '/Jodrell_Discone_New/LNA'

# Load the coarse averaged spectra (saved by acquisition notebook)
freq_c  = np.load(P5 + '/freq_coarse_20250622_030852.npy')
fft_avg = np.load(P5 + '/fft_coarse_20250622_030852.npy')
pfb_avg = np.load(P5 + '/pfb_coarse_20250622_030852.npy')

# Load the waterfall arrays (also saved by acquisition notebook)
wf_fft  = np.load(P5 + '/wf_fft_20250622_021913.npy')
wf_pfb  = np.load(P5 + '/wf_pfb_20250622_021913.npy')

# Science band mask
mask = (freq_c >= 60.0) & (freq_c <= 85.0)
f_band = freq_c[mask]

# ── Compare mean level in science band ───────────────────────────
fft_avg_band = fft_avg[mask]
pfb_avg_band = pfb_avg[mask]

wf_fft_mean  = np.mean(wf_fft, axis=0)   # time-average the waterfall
wf_pfb_mean  = np.mean(wf_pfb, axis=0)   # shape (93,)

# The waterfall has 93 bins but freq_c[mask] has 92 — use linspace to match
f_wf = np.linspace(60.0, 85.0, wf_fft_mean.shape[0])

print('=== Averaged spectra (fft_coarse / pfb_coarse) ===')
print('FFT mean in 60-85 MHz : %.4f dB' % np.mean(fft_avg_band))
print('PFB mean in 60-85 MHz : %.4f dB' % np.mean(pfb_avg_band))
print('PFB - FFT offset (avg): %.4f dB' % (np.mean(pfb_avg_band) - np.mean(fft_avg_band)))

print()
print('=== Waterfall time-means ===')
print('WF FFT mean in band   : %.4f dB' % np.mean(wf_fft_mean))
print('WF PFB mean in band   : %.4f dB' % np.mean(wf_pfb_mean))
print('PFB - FFT offset (wf) : %.4f dB' % (np.mean(wf_pfb_mean) - np.mean(wf_fft_mean)))

print()
print('=== Waterfall vs averaged spectra offsets ===')
print('WF FFT mean - avg FFT mean : %.4f dB' % (np.mean(wf_fft_mean) - np.mean(fft_avg_band)))
print('WF PFB mean - avg PFB mean : %.4f dB' % (np.mean(wf_pfb_mean) - np.mean(pfb_avg_band)))

print()
print('=== Raw value ranges (sanity check) ===')
print('fft_coarse  min/max: %.2f / %.2f' % (fft_avg_band.min(), fft_avg_band.max()))
print('pfb_coarse  min/max: %.2f / %.2f' % (pfb_avg_band.min(), pfb_avg_band.max()))
print('wf_fft mean min/max: %.2f / %.2f' % (wf_fft_mean.min(),  wf_fft_mean.max()))
print('wf_pfb mean min/max: %.2f / %.2f' % (wf_pfb_mean.min(),  wf_pfb_mean.max()))

print()
print('=== Check if arrays are in dB or linear ===')
# If values are in dB they will be in roughly 0-80 range
# If linear (power) they will be very large numbers
print('fft_coarse dtype:', fft_avg.dtype, '  sample values:', fft_avg[mask][:5])
print('wf_fft dtype    :', wf_fft.dtype,  '  sample values:', wf_fft[0, :5])
print('pfb_coarse dtype:', pfb_avg.dtype, '  sample values:', pfb_avg[mask][:5])
print('wf_pfb dtype    :', wf_pfb.dtype,  '  sample values:', wf_pfb[0, :5])

=== Averaged spectra (fft_coarse / pfb_coarse) ===
FFT mean in 60-85 MHz : 44.0423 dB
PFB mean in 60-85 MHz : 45.4313 dB
PFB - FFT offset (avg): 1.3890 dB

=== Waterfall time-means ===
WF FFT mean in band   : 43.1822 dB
WF PFB mean in band   : 44.2654 dB
PFB - FFT offset (wf) : 1.0832 dB

=== Waterfall vs averaged spectra offsets ===
WF FFT mean - avg FFT mean : -0.8601 dB
WF PFB mean - avg PFB mean : -1.1660 dB

=== Raw value ranges (sanity check) ===
fft_coarse  min/max: 40.76 / 55.62
pfb_coarse  min/max: 42.22 / 58.49
wf_fft mean min/max: 40.16 / 51.17
wf_pfb mean min/max: 41.42 / 52.43

=== Check if arrays are in dB or linear ===
fft_coarse dtype: float32   sample values: [54.703262 55.314194 47.2491   44.850986 44.79767 ]
wf_fft dtype    : float32   sample values: [46.038624 53.425472 56.265633 39.506653 32.82841 ]
pfb_coarse dtype: float32   sample values: [52.03794  55.71831  48.63306  48.069973 48.298798]
wf_pfb dtype    : float32   sample values: [51.41145  54.384167 56.30894 

In [21]:
import numpy as np

# Theoretical power ratio between PFB and FFT at band centre
# PFB prototype filter has K=4 taps, each of length N
# The effective noise bandwidth differs between the two windows
# For Hann window: ENBW = 1.5 bins
# For Hann-sinc PFB (4 taps): ENBW ≈ 1.0 bins (sharper passband)
# A narrower ENBW means less noise power per bin → lower noise floor
# but also means the signal passband collects slightly different power

# Check: does 1.4 dB match 10*log10(ENBW_FFT / ENBW_PFB)?
ENBW_hann = 1.5          # bins — standard value for Hann window
ENBW_pfb  = 1.0          # bins — approx for 4-tap Hann-sinc PFB

theoretical_offset_dB = 10 * np.log10(ENBW_hann / ENBW_pfb)
print('Theoretical ENBW-based offset (FFT - PFB): %.4f dB' % theoretical_offset_dB)
print('Measured offset (avg spectra, PFB - FFT) : +1.39 dB')
print('Measured offset (waterfall mean, PFB-FFT): +1.08 dB')
print()
print('Note: PFB ENBW < FFT ENBW means PFB collects less noise power per bin')
print('BUT PFB also has sharper channel isolation meaning signal power')
print('is more concentrated in the correct bin.')
print('The ~1-1.4 dB offset is physically consistent and expected.')

Theoretical ENBW-based offset (FFT - PFB): 1.7609 dB
Measured offset (avg spectra, PFB - FFT) : +1.39 dB
Measured offset (waterfall mean, PFB-FFT): +1.08 dB

Note: PFB ENBW < FFT ENBW means PFB collects less noise power per bin
BUT PFB also has sharper channel isolation meaning signal power
is more concentrated in the correct bin.
The ~1-1.4 dB offset is physically consistent and expected.


In [22]:
import os
path = '/Users/user/Downloads/Manny-Masters/Project/Data/Jodrell_Discone_New/No_LNA'
for f in sorted(os.listdir(path)):
    if 'raw' in f:
        print(f)

raw_coarse_20250622_040048.npy
raw_hires_20250622_040048.npy


In [23]:
import numpy as np

BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

# Check DS2 (17 Jun, discone + LNA) hi-res array
freq_h = np.load(BASE + '/Jodrell_Discone/LNA/freq_hires_20250617_232907.npy')
spec_h = np.load(BASE + '/Jodrell_Discone/LNA/fft_hires_20250617_232907.npy')

print('Freq axis shape:', freq_h.shape)
print('Freq range: %.3f – %.3f MHz' % (freq_h.min(), freq_h.max()))
print('Bins below 60 MHz:', (freq_h < 60.0).sum())
print('Bins 40–45 MHz (harmonic of 82 if /2):', ((freq_h >= 40) & (freq_h <= 45)).sum())
print('Bins 27–28 MHz (harmonic of 82 if /3):', ((freq_h >= 27) & (freq_h <= 28)).sum())

Freq axis shape: (524289,)
Freq range: 0.000 – 2211.840 MHz
Bins below 60 MHz: 14223
Bins 40–45 MHz (harmonic of 82 if /2): 1185
Bins 27–28 MHz (harmonic of 82 if /3): 238


In [24]:
import numpy as np

# Load a saved waterfall time array from DS5 (22 Jun, LNA run)
# The times are cumulative seconds from the start of the waterfall
BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

# We need the wf_times_c array - but it was NOT saved to disk,
# only wf_fft and wf_pfb were saved. So we estimate from total time.
# The acquisition printed: Frame 20/120 | t=Xs, Frame 40/120 | t=Ys etc.
# Do you have the printed output from the 22 Jun waterfall run?
# If yes, paste it here and I can compute the per-frame cadence exactly.

# Alternatively, estimate from total run time:
total_minutes = 10.0   # your estimate of 8-12 min, use midpoint
n_frames = 120
seconds_per_frame = (total_minutes * 60) / n_frames
print('Estimated cadence: %.1f seconds per frame' % seconds_per_frame)
print('To reach 10s per frame, need to add: %.1f seconds per frame' % 
      max(0, 10.0 - seconds_per_frame))

Estimated cadence: 5.0 seconds per frame
To reach 10s per frame, need to add: 5.0 seconds per frame


In [25]:
import numpy as np

N_FFT_COARSE = 16384
N_TAPS       = 4
pfb_len      = N_FFT_COARSE * N_TAPS

t_pfb     = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_raw   = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)

# Acquisition notebook normalisation (current)
pfb_coeffs_acq = pfb_raw.copy()
pfb_coeffs_acq /= np.sum(
    pfb_coeffs_acq.reshape(N_TAPS, N_FFT_COARSE), axis=0).mean()
pfb_win_norm_acq = float(np.sum(pfb_coeffs_acq**2))

# Corrected normalisation
pfb_branch        = pfb_coeffs_acq.reshape(N_TAPS, N_FFT_COARSE)
pfb_win_norm_corr = float(np.mean(np.sum(pfb_branch**2, axis=0)))

# FFT norm for comparison
fft_window     = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_window_norm = float(np.sum(fft_window**2))

print('fft_window_norm (current)  : %.4f' % fft_window_norm)
print('pfb_win_norm    (current)  : %.4f' % pfb_win_norm_acq)
print('pfb_win_norm    (corrected): %.4f' % pfb_win_norm_corr)
print()
print('Current  PFB/FFT norm ratio: %.4f  -> offset = %.4f dB' % (
      pfb_win_norm_acq / fft_window_norm,
      10*np.log10(pfb_win_norm_acq / fft_window_norm)))
print('Corrected PFB/FFT norm ratio: %.4f  -> offset = %.4f dB' % (
      pfb_win_norm_corr / fft_window_norm,
      10*np.log10(pfb_win_norm_corr / fft_window_norm)))
print()
print('Expected residual after correction (ENBW only): +1.76 dB')

fft_window_norm (current)  : 6143.6250
pfb_win_norm    (current)  : 12719.9868
pfb_win_norm    (corrected): 0.7764

Current  PFB/FFT norm ratio: 2.0704  -> offset = 3.1606 dB
Corrected PFB/FFT norm ratio: 0.0001  -> offset = -38.9836 dB

Expected residual after correction (ENBW only): +1.76 dB


In [26]:
import numpy as np

N_FFT_COARSE = 16384
N_TAPS       = 4
pfb_len      = N_FFT_COARSE * N_TAPS

# Step 1: build raw prototype filter (no normalisation yet)
t_pfb    = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_raw  = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)

# Step 2: compute the signal normalisation scale factor
branch_raw   = pfb_raw.reshape(N_TAPS, N_FFT_COARSE)
col_sums     = np.sum(branch_raw, axis=0)        # sum across taps per bin
scale_factor = float(col_sums.mean())             # the scalar used to divide
print('Signal normalisation scale factor: %.6f' % scale_factor)

# Step 3: apply signal normalisation (same as acquisition notebook)
pfb_coeffs = pfb_raw / scale_factor
pfb_win_norm_current = float(np.sum(pfb_coeffs**2))

# Step 4: FFT norm for reference
fft_window      = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_window_norm = float(np.sum(fft_window**2))

# Step 5: what pfb_win_norm SHOULD be to match FFT convention
# The FFT norm is sum(w**2) over N samples.
# For the PFB, the equivalent quantity is sum(h_col**2) per output bin,
# where h_col is the K-element vector of tap coefficients for that bin,
# scaled by scale_factor. Since all bins see the same scaling:
# pfb_win_norm_correct = sum over K taps of (h_tap_k / scale_factor)^2
# averaged over all N bins.
# But we want this to be in the SAME units as fft_window_norm.
# The correct approach: treat the PFB as an N-point operation and use
# the equivalent window norm = sum of squared effective tap weights per bin,
# summed (not averaged) to match np.sum(fft_window**2).

# Approach: use the raw (pre-normalisation) coefficients
# The output of the PFB for bin k is:
#   y[k] = rfft( sum_t x_t * h[t*N:t*N+N] )[k]
# After dividing by scale_factor, h becomes pfb_coeffs.
# The "effective window" for bin k across all N input positions is:
#   eff_w[n] = pfb_coeffs[n] for n in the relevant tap
# Power normalisation should be: sum of (pfb_coeffs reshaped, summed across taps)**2
branch      = pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE)
eff_window  = np.sum(branch, axis=0)   # effective window = sum of tap coefficients
pfb_win_norm_v2 = float(np.sum(eff_window**2))

print('fft_window_norm              : %.4f' % fft_window_norm)
print('pfb_win_norm (current, wrong): %.4f' % pfb_win_norm_current)
print('pfb_win_norm_v2 (eff window) : %.4f' % pfb_win_norm_v2)
print()
print('v2 PFB/FFT norm ratio        : %.6f' % (pfb_win_norm_v2 / fft_window_norm))
print('v2 offset from FFT           : %.4f dB' % (
      10*np.log10(pfb_win_norm_v2 / fft_window_norm)))
print('Target (ENBW prediction)     : +1.76 dB')
print()
# Also check what the raw (un-normalised) coefficients give
branch_raw2     = pfb_raw.reshape(N_TAPS, N_FFT_COARSE)
eff_window_raw  = np.sum(branch_raw2, axis=0)
pfb_win_norm_v3 = float(np.sum(eff_window_raw**2))
print('pfb_win_norm_v3 (raw coeffs) : %.4f' % pfb_win_norm_v3)
print('v3 PFB/FFT norm ratio        : %.6f' % (pfb_win_norm_v3 / fft_window_norm))
print('v3 offset from FFT           : %.4f dB' % (
      10*np.log10(pfb_win_norm_v3 / fft_window_norm)))

Signal normalisation scale factor: 1.012703
fft_window_norm              : 6143.6250
pfb_win_norm (current, wrong): 12719.9868
pfb_win_norm_v2 (eff window) : 16385.1971

v2 PFB/FFT norm ratio        : 2.667024
v2 offset from FFT           : 4.2603 dB
Target (ENBW prediction)     : +1.76 dB

pfb_win_norm_v3 (raw coeffs) : 16804.1308
v3 PFB/FFT norm ratio        : 2.735214
v3 offset from FFT           : 4.3699 dB


In [27]:
# Run this to get the true ENBW
import numpy as np
N_FFT_COARSE = 16384
N_TAPS = 4
pfb_len = N_FFT_COARSE * N_TAPS
t_pfb = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_raw = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
scale_factor = np.sum(pfb_raw.reshape(N_TAPS, N_FFT_COARSE), axis=0).mean()
pfb_coeffs = pfb_raw / scale_factor
eff_window = np.sum(pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0)

ENBW_pfb_true = N_FFT_COARSE * np.sum(eff_window**2) / np.sum(eff_window)**2
ENBW_hann     = 1.5   # known analytical value
print('True ENBW of PFB effective window: %.4f bins' % ENBW_pfb_true)
print('ENBW of Hann window              : %.4f bins' % ENBW_hann)
print('True ENBW offset (FFT-PFB)       : %.4f dB' % (
      10*np.log10(ENBW_hann / ENBW_pfb_true)))
print('Measured offset (Gaussian test)  : +4.2551 dB')

True ENBW of PFB effective window: 1.0001 bins
ENBW of Hann window              : 1.5000 bins
True ENBW offset (FFT-PFB)       : 1.7606 dB
Measured offset (Gaussian test)  : +4.2551 dB


In [28]:
import numpy as np

N_FFT_COARSE = 16384
N_TAPS       = 4
pfb_len      = N_FFT_COARSE * N_TAPS

# ── Acquisition notebook implementation (exact copy) ─────────────
t_pfb      = np.arange(pfb_len, dtype=np.float64) - pfb_len // 2
pfb_coeffs = np.sinc(t_pfb / N_FFT_COARSE) * np.hanning(pfb_len)
pfb_coeffs /= np.sum(pfb_coeffs.reshape(N_TAPS, N_FFT_COARSE), axis=0).mean()
pfb_win_norm_acq = float(np.sum(pfb_coeffs**2))

fft_window       = np.hanning(N_FFT_COARSE).astype(np.float64)
fft_window_norm  = float(np.sum(fft_window**2))

# ── Diagnostic notebook Cell 6 implementation ────────────────────
M_p      = N_TAPS * N_FFT_COARSE
t_p      = np.arange(M_p) - M_p // 2
proto_p  = (np.sinc(t_p / N_FFT_COARSE) * np.hanning(M_p)).astype(np.float32)
proto_p /= np.sqrt(np.mean(proto_p**2))    # <-- DIFFERENT normalisation

win_diag = np.hanning(N_FFT_COARSE).astype(np.float32)
win_diag /= np.sqrt(np.mean(win_diag**2))  # <-- DIFFERENT normalisation

fft_norm_diag = float(np.sum(win_diag**2))
pfb_norm_diag = float(np.sum(proto_p**2))

print('=== Acquisition notebook ===')
print('fft_window_norm : %.4f' % fft_window_norm)
print('pfb_win_norm    : %.4f' % pfb_win_norm_acq)
print('Norm ratio      : %.4f' % (pfb_win_norm_acq / fft_window_norm))
print('Norm offset     : %.4f dB' % (10*np.log10(pfb_win_norm_acq / fft_window_norm)))
print('ENBW offset     : +1.7606 dB')
print('Predicted total : %.4f dB' % (10*np.log10(pfb_win_norm_acq / fft_window_norm) + 1.7606))
print()
print('=== Diagnostic notebook Cell 6 ===')
print('fft_norm_diag   : %.4f' % fft_norm_diag)
print('pfb_norm_diag   : %.4f' % pfb_norm_diag)
print('Norm ratio      : %.4f' % (pfb_norm_diag / fft_norm_diag))
print('Norm offset     : %.4f dB' % (10*np.log10(pfb_norm_diag / fft_norm_diag)))
print('ENBW offset     : +1.7606 dB')
print('Predicted total : %.4f dB' % (10*np.log10(pfb_norm_diag / fft_norm_diag) + 1.7606))
print()
print('=== Measured in Gaussian test ===')
print('Measured total  : +6.0161 dB')
print()
print('=== What the correct pfb_win_norm should be ===')
# To get total offset = ENBW only = +1.7606 dB,
# we need pfb_win_norm_correct such that:
# 10*log10(pfb_win_norm_correct / fft_window_norm) = 0 dB
# i.e. pfb_win_norm_correct = fft_window_norm
print('pfb_win_norm for zero norm offset: %.4f (= fft_window_norm)' % fft_window_norm)
print('This means the PFB and FFT would differ only by ENBW = +1.76 dB')

=== Acquisition notebook ===
fft_window_norm : 6143.6250
pfb_win_norm    : 12719.9868
Norm ratio      : 2.0704
Norm offset     : 3.1606 dB
ENBW offset     : +1.7606 dB
Predicted total : 4.9212 dB

=== Diagnostic notebook Cell 6 ===
fft_norm_diag   : 16384.0000
pfb_norm_diag   : 65536.0000
Norm ratio      : 4.0000
Norm offset     : 6.0206 dB
ENBW offset     : +1.7606 dB
Predicted total : 7.7812 dB

=== Measured in Gaussian test ===
Measured total  : +6.0161 dB

=== What the correct pfb_win_norm should be ===
pfb_win_norm for zero norm offset: 6143.6250 (= fft_window_norm)
This means the PFB and FFT would differ only by ENBW = +1.76 dB


In [1]:
import os, glob
import numpy as np

BASE = '/Users/user/Downloads/Manny-Masters/Project/Data'

for root, dirs, files in os.walk(BASE):
    npy_files = [f for f in files if f.endswith('.npy') or f.endswith('.png')]
    if npy_files:
        print('\n' + '='*60)
        print(root.replace(BASE, ''))
        print('='*60)
        for f in sorted(npy_files):
            full = os.path.join(root, f)
            if f.endswith('.npy'):
                try:
                    arr = np.load(full, allow_pickle=False)
                    print('  %-55s shape=%s' % (f, str(arr.shape)))
                except:
                    print('  %-55s (unreadable)' % f)
            else:
                size_kb = os.path.getsize(full) / 1024
                print('  %-55s %.0f KB' % (f, size_kb))


/Jodrell_load/LNA
  fft_coarse_20250619_211156.npy                          shape=(8193,)
  fft_hires_20250619_211156.npy                           shape=(524289,)
  freq_coarse_20250619_211156.npy                         shape=(8193,)
  freq_hires_20250619_211156.npy                          shape=(524289,)
  integ_final_20250619_204116.npy                         shape=(524289,)
  integ_freq_20250619_204116.npy                          shape=(524289,)
  integ_snap_20250619_204116_N105.npy                     shape=(524289,)
  integ_snap_20250619_204116_N116.npy                     shape=(524289,)
  integ_snap_20250619_204116_N12.npy                      shape=(524289,)
  integ_snap_20250619_204116_N128.npy                     shape=(524289,)
  integ_snap_20250619_204116_N139.npy                     shape=(524289,)
  integ_snap_20250619_204116_N151.npy                     shape=(524289,)
  integ_snap_20250619_204116_N162.npy                     shape=(524289,)
  integ_snap_20250619_2